# Resting-state Hurst statistics across MRS voxels

In [3]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from statsmodels.stats.multitest import multipletests

## 1. Configuration

In [4]:
repo_root = next(
    (
        path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "CODE").is_dir() and (path / "DATA").is_dir()
    ),
    None,
)

hurst_path = (
    repo_root / "RESULTS/discovery_dataset/hurst/dataframes"
    / "df_hurst_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_w_rest_IQR_cleaned.csv"
)
selected_parcels_path = repo_root / "DATA/discovery_dataset/selected_parcels_overlap30.csv"
output_dir = repo_root / "RESULTS/discovery_dataset/statistics/Hurst_repeated_measures"

value_col = "Hursts"
alpha = 0.05
voxel_order = ["anteriorcingulate", "occipital", "rightauditory", "smleft"]

for path in (hurst_path, selected_parcels_path):
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found: {path}")
output_dir.mkdir(parents=True, exist_ok=True)

## 2. Subject-level mean H by voxel

In [5]:
hurst_df = pd.read_csv(hurst_path)
selected_df = pd.read_csv(selected_parcels_path)

required = {"Subject", "Parcel", value_col}
if missing := required.difference(hurst_df.columns):
    raise ValueError(f"Missing columns in Hurst data: {sorted(missing)}")

# Rest is the intended input; filter explicitly if the CSV also contains other tasks.
if "Task" in hurst_df.columns:
    rest_mask = hurst_df["Task"].astype("string").str.strip().str.lower().eq("rest")
    hurst_df = hurst_df.loc[rest_mask.fillna(False)].copy()

selected_df.columns = selected_df.columns.astype(str).str.strip().str.lower()
parcel_col = "parcel_id" if "parcel_id" in selected_df else "parcel"
if "voxel" not in selected_df or parcel_col not in selected_df:
    raise ValueError("The selected-parcels CSV needs 'Voxel' and 'parcel_id' (or 'Parcel').")

selected_df = selected_df[["voxel", parcel_col]].rename(
    columns={"voxel": "Voxel", parcel_col: "Parcel"}
)
selected_df["Voxel"] = selected_df["Voxel"].astype("string").str.strip().str.lower()
selected_df = (
    selected_df.loc[selected_df["Voxel"].isin(voxel_order)]
    .dropna(subset=["Voxel", "Parcel"])
    .drop_duplicates(subset=["Voxel", "Parcel"])
    .copy()
)

hurst_df = hurst_df[["Subject", "Parcel", value_col]].dropna(
    subset=["Subject", "Parcel"]
).copy()
for data, label in ((hurst_df, "Hurst data"), (selected_df, "parcel lookup")):
    ids = pd.to_numeric(data["Parcel"], errors="raise")
    if not (np.isfinite(ids) & (ids % 1 == 0)).all():
        raise ValueError(f"{label} contains invalid parcel IDs.")
    data["Parcel"] = ids.astype(int)

if hurst_df.duplicated(subset=["Subject", "Parcel"]).any():
    raise ValueError("Duplicate Subject–Parcel entries: check that the CSV is rest-only.")

hurst_df[value_col] = pd.to_numeric(hurst_df[value_col], errors="coerce")
hurst_df[value_col] = hurst_df[value_col].replace([np.inf, -np.inf], np.nan)

merged = hurst_df.merge(selected_df, on="Parcel", how="inner")
subject_voxel_hurst = (
    merged.groupby(["Subject", "Voxel"], as_index=False)[value_col]
    .mean()
    .dropna(subset=[value_col])
)
if subject_voxel_hurst.empty:
    raise ValueError("No valid Hurst values match the selected voxel parcels.")

wide = subject_voxel_hurst.pivot(index="Subject", columns="Voxel", values=value_col)
missing_voxels = set(voxel_order).difference(wide.columns)
if missing_voxels:
    raise ValueError(f"No observations available for voxels: {sorted(missing_voxels)}")
wide_complete = wide.reindex(columns=voxel_order).dropna().copy()
if len(wide_complete) < 3:
    raise ValueError("At least 3 complete subjects are required for the normality checks.")

complete_long = wide_complete.reset_index().melt(
    id_vars="Subject", value_vars=voxel_order, var_name="Voxel", value_name=value_col
)
subject_voxel_hurst.to_csv(output_dir / "subject_voxel_mean_hurst.csv", index=False)
complete_long.to_csv(output_dir / "subject_voxel_mean_hurst_complete_long.csv", index=False)
wide_complete.to_csv(output_dir / "subject_voxel_mean_hurst_complete_wide.csv")

n_subjects_total = subject_voxel_hurst["Subject"].nunique()
print(f"Subjects with at least one voxel: {n_subjects_total}")
print(f"Subjects with all four voxels: {len(wide_complete)}")
print(f"Excluded from repeated-measures tests: {n_subjects_total - len(wide_complete)}")

Subjects with at least one voxel: 164
Subjects with all four voxels: 163
Excluded from repeated-measures tests: 1


## 3. Descriptive statistics and Shapiro–Wilk diagnostics

In [6]:
descriptive_rows = []
normality_rows = []
for voxel in voxel_order:
    values = wide_complete[voxel].to_numpy(dtype=float)
    shapiro_stat, shapiro_p = stats.shapiro(values)
    descriptive_rows.append({
        "Voxel": voxel,
        "N_subjects": len(values),
        "Mean": float(np.mean(values)),
        "SD": float(np.std(values, ddof=1)),
        "Median": float(np.median(values)),
        "Min": float(np.min(values)),
        "Max": float(np.max(values)),
    })
    normality_rows.append({
        "Voxel": voxel,
        "Test": "Shapiro–Wilk",
        "Statistic": float(shapiro_stat),
        "P": float(shapiro_p),
        "Pass_p_gt_0.05": bool(np.isfinite(shapiro_p) and shapiro_p > alpha),
        "N": len(values),
    })

descriptive_df = pd.DataFrame(descriptive_rows)
normality_df = pd.DataFrame(normality_rows)
descriptive_df.to_csv(output_dir / "descriptive_statistics_by_voxel.csv", index=False)
normality_df.to_csv(output_dir / "normality_by_voxel.csv", index=False)

display(descriptive_df)
display(normality_df)

,Voxel,N_subjects,Mean,SD,Median,Min,Max
0,anteriorcingulate,163,1.214807,0.137575,1.227013,0.804468,1.539801
1,occipital,163,1.397668,0.150425,1.409901,0.953696,1.747212
2,rightauditory,163,1.216433,0.176415,1.198970,0.788338,1.653568
3,smleft,163,1.388818,0.199358,1.398972,0.909365,1.855951


,Voxel,Test,Statistic,P,Pass_p_gt_0.05,N
0,anteriorcingulate,Shapiro–Wilk,0.988424,0.200234,True,163
1,occipital,Shapiro–Wilk,0.989475,0.266320,True,163
2,rightauditory,Shapiro–Wilk,0.992304,0.535326,True,163
3,smleft,Shapiro–Wilk,0.987359,0.148907,True,163


## 4. Friedman omnibus test

In [7]:
n_subjects, n_voxels = wide_complete.shape
friedman_stat, friedman_p = stats.friedmanchisquare(
    *(wide_complete[voxel].to_numpy(dtype=float) for voxel in voxel_order)
)
omnibus_df = pd.DataFrame([{
    "Chosen_test": "Friedman test",
    "Statistic": float(friedman_stat),
    "P": float(friedman_p),
    "N_subjects": n_subjects,
    "N_conditions": n_voxels,
    "Kendalls_W": float(friedman_stat / (n_subjects * (n_voxels - 1))),
}])
omnibus_df.to_csv(output_dir / "omnibus_repeated_measures_test.csv", index=False)
display(omnibus_df)

,Chosen_test,Statistic,P,N_subjects,N_conditions,Kendalls_W
0,Friedman test,232.376687,4.235814e-50,163,4,0.475208


## 5. Paired Wilcoxon tests with BH-FDR

In [8]:
pairwise_rows = []
for voxel1, voxel2 in combinations(voxel_order, 2):
    x = wide_complete[voxel1].to_numpy(dtype=float)
    y = wide_complete[voxel2].to_numpy(dtype=float)
    raw_differences = x - y
    differences = np.round(raw_differences, decimals=12)
    n_effective = int(np.count_nonzero(differences))

    if n_effective == 0:
        statistic = p_raw = z_value = effect_r = np.nan
    else:
        test = stats.wilcoxon(
            differences,
            alternative="two-sided",
            zero_method="wilcox",
            method="approx",
            correction=False,
        )
        statistic = float(test.statistic)
        p_raw = float(test.pvalue)
        z_value = float(test.zstatistic)
        effect_r = abs(z_value) / np.sqrt(n_effective)

    median_difference = float(np.median(raw_differences))
    direction = (
        f"{voxel1} > {voxel2}" if median_difference > 0 else
        f"{voxel1} < {voxel2}" if median_difference < 0 else
        "equal median difference"
    )
    magnitude = (
        np.nan if np.isnan(effect_r) else
        "negligible" if effect_r < 0.10 else
        "small" if effect_r < 0.30 else
        "medium" if effect_r < 0.50 else "large"
    )
    pairwise_rows.append({
        "Voxel1": voxel1,
        "Voxel2": voxel2,
        "Test": "Wilcoxon signed-rank",
        "Wilcoxon_statistic": statistic,
        "Wilcoxon_Z": z_value,
        "P_raw": p_raw,
        "Mean_Voxel1": float(np.mean(x)),
        "Mean_Voxel2": float(np.mean(y)),
        "Median_Voxel1": float(np.median(x)),
        "Median_Voxel2": float(np.median(y)),
        "MeanDiff_Voxel1_minus_Voxel2": float(np.mean(raw_differences)),
        "MedianDiff_Voxel1_minus_Voxel2": median_difference,
        "Direction": direction,
        "Wilcoxon_effect_size_r": effect_r,
        "Effect_size_magnitude": magnitude,
        "N_total_pairs": len(x),
        "N_nonzero_differences": n_effective,
    })

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_df["P_FDR_BH"] = np.nan
pairwise_df["Reject_FDR_0.05"] = False
valid = pairwise_df["P_raw"].notna()
if valid.any():
    reject, adjusted_p, _, _ = multipletests(
        pairwise_df.loc[valid, "P_raw"], alpha=alpha, method="fdr_bh"
    )
    pairwise_df.loc[valid, "P_FDR_BH"] = adjusted_p
    pairwise_df.loc[valid, "Reject_FDR_0.05"] = reject
pairwise_df = pairwise_df.sort_values("P_FDR_BH", na_position="last").reset_index(drop=True)

pairwise_df.to_csv(output_dir / "pairwise_repeated_measures_FDR.csv", index=False)
display(pairwise_df[[
    "Voxel1", "Voxel2", "MedianDiff_Voxel1_minus_Voxel2", "Direction",
    "Wilcoxon_statistic", "Wilcoxon_Z", "Wilcoxon_effect_size_r",
    "Effect_size_magnitude", "P_raw", "P_FDR_BH", "Reject_FDR_0.05",
    "N_total_pairs", "N_nonzero_differences",
]])
print(f"Statistics saved to: {output_dir}")

,Voxel1,Voxel2,MedianDiff_Voxel1_minus_Voxel2,Direction,Wilcoxon_statistic,Wilcoxon_Z,Wilcoxon_effect_size_r,Effect_size_magnitude,P_raw,P_FDR_BH,Reject_FDR_0.05,N_total_pairs,N_nonzero_differences
0,anteriorcingulate,occipital,-0.195256,anteriorcingulate < occipital,218.0,-10.712345,0.839056,large,8.907650e-27,5.344590e-26,True,163,163
1,occipital,rightauditory,0.163422,occipital > rightauditory,496.0,-10.251706,0.802976,large,1.162732e-24,3.488196e-24,True,163,163
2,rightauditory,smleft,-0.168941,rightauditory < smleft,722.0,-9.877229,0.773644,large,5.225778e-23,1.045156e-22,True,163,163
3,anteriorcingulate,smleft,-0.178918,anteriorcingulate < smleft,929.0,-9.534236,0.746779,large,1.509939e-21,2.264909e-21,True,163,163
4,occipital,smleft,-0.002962,occipital < smleft,6402.0,-0.465610,0.036469,negligible,6.414947e-01,7.697936e-01,False,163,163
5,anteriorcingulate,rightauditory,0.006751,anteriorcingulate > rightauditory,6528.0,-0.256831,0.020117,negligible,7.973091e-01,7.973091e-01,False,163,163


Statistics saved to: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/RESULTS/discovery_dataset/statistics/Hurst_repeated_measures
